# 08_wrapping_up/08_wrapping_up.ipynb

# Chapter 8: Wrapping Up & Modern ML Workflows

## Overview
This notebook synthesizes the entire machine learning engineering lifecycle into a production-ready framework:
1. **Baseline Comparisons & Sanity Checks**: Establishing performance benchmarks using `DummyClassifier` and `DummyRegressor`.
2. **End-to-End Production Pipeline**: Constructing an integrated pipeline handling heterogeneous data types, scaling, encoding, model selection, and hyperparameter tuning.
3. **Error Analysis & Model Diagnostics**: Inspecting misclassified samples, residuals, and feature importances to guide model refinement.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics: ClassificationReportDisplay, ConfusionMatrixDisplay, classification_report, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

# Matplotlib global settings
plt.rc("font", size=10)
plt.rc("axes", labelsize=11, titlesize=12)

## 1. Baseline Comparisons & Sanity Checks

Before deploying complex non-linear algorithms, always establish simple **baseline estimators**:
- **`DummyClassifier`**: Uses simple heuristics (e.g., `strategy="most_frequent"` or `"stratified"`) to provide a lower-bound performance threshold.
- **`DummyRegressor`**: Predicts central metrics (e.g., `strategy="mean"` or `"median"`).

If a complex model fails to significantly outperform a dummy baseline, the task may lack signal, features may require re-engineering, or target labels may be randomly distributed.

In [ ]:
# Load breast cancer dataset for classification sanity check
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c
)

# 1. Dummy Baseline (Predict Most Frequent Class)
dummy_clf = DummyClassifier(strategy="most_frequent").fit(X_tr_c, y_tr_c)
dummy_acc = dummy_clf.score(X_te_c, y_te_c)

# 2. Trained ML Estimator
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr_c, y_tr_c)
rf_acc = rf_clf.score(X_te_c, y_te_c)

print("--- Classification Sanity Check ---")
print(f"Dummy Classifier (Most Frequent Baseline): {dummy_acc * 100:.2f}%")
print(f"Random Forest Classifier Accuracy:          {rf_acc * 100:.2f}%")
print(f"Performance Gain over Baseline:             {(rf_acc - dummy_acc) * 100:.2f}%")

# Load housing dataset for regression sanity check
housing = fetch_california_housing()
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(
    housing.data, housing.target, test_size=0.25, random_state=42
)

# 1. Dummy Regressor (Predict Mean)
dummy_reg = DummyRegressor(strategy="mean").fit(X_tr_h, y_tr_h)
dummy_r2 = dummy_reg.score(X_te_h, y_te_h)

print("\n--- Regression Sanity Check ---")
print(f"Dummy Regressor (Mean Baseline R²): {dummy_r2:.4f}")

## 2. End-to-End Integrated Machine Learning Pipeline

A full production pipeline combines:
1. **Preprocessing Layer**: Heterogeneous column transformations via `ColumnTransformer`.
2. **Estimator Layer**: Scalable non-linear algorithms (e.g., `HistGradientBoostingClassifier`).
3. **Optimization Layer**: Cross-validation hyperparameter search via `GridSearchCV`.

In [ ]:
# Construct synthetic heterogeneous dataset with numerical, categorical, and missing values
np.random.seed(42)
n_samples = 1000

df_synthetic = pd.DataFrame(
    {
        "Age": np.random.randint(18, 70, size=n_samples).astype(float),
        "Income": np.random.exponential(scale=50000, size=n_samples),
        "CreditScore": np.random.normal(loc=650, scale=100, size=n_samples),
        "Education": np.random.choice(
            ["HighSchool", "Bachelor", "Master", "PhD"], size=n_samples
        ),
        "EmploymentStatus": np.random.choice(
            ["Employed", "Unemployed", "SelfEmployed"], size=n_samples
        ),
    }
)

# Generate synthetic binary target
y_synthetic = (
    (df_synthetic["Income"] > 40000) & (df_synthetic["CreditScore"] > 600)
).astype(int)

# Split data into train and test sets
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    df_synthetic, y_synthetic, test_size=0.2, random_state=42, stratify=y_synthetic
)

# Define column groups
num_cols = ["Age", "Income", "CreditScore"]
cat_cols = ["Education", "EmploymentStatus"]

# Preprocessing specification
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ]
)

# Pipeline assembly
full_pipeline = Pipeline(
    [("preprocessor", preprocessor), ("classifier", HistGradientBoostingClassifier(random_state=42))]
)

# Hyperparameter optimization grid
param_grid = {
    "classifier__learning_rate": [0.01, 0.1],
    "classifier__max_iter": [50, 100],
    "classifier__max_depth": [3, 5],
}

grid_prod = GridSearchCV(full_pipeline, param_grid=param_grid, cv=5, n_jobs=-1)
grid_prod.fit(X_train_s, y_train_s)

print("Best Parameters Found:")
for k, v in grid_prod.best_params_.items():
    print(f"  {k}: {v}")

print(f"\nBest Cross-Validation Score: {grid_prod.best_score_ * 100:.2f}%")
print(f"Hold-out Test Set Score:     {grid_prod.score(X_test_s, y_test_s) * 100:.2f}%")

## 3. Model Error Analysis & Diagnostics

Evaluating aggregate metrics (such as accuracy or $R^2$) is insufficient for deployment.
**Error Analysis** involves inspecting individual misclassifications, analyzing false positives/negatives, and auditing model predictions using diagnostic visual tools.

In [ ]:
y_pred_prod = grid_prod.predict(X_test_s)

# Display Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ConfusionMatrixDisplay.from_predictions(
    y_test_s,
    y_pred_prod,
    display_labels=["Denied", "Approved"],
    ax=axes[0],
    cmap="Blues",
)
axes[0].set_title("Test Set Confusion Matrix")

# Locate misclassified indices for deep inspection
misclassified_mask = y_test_s != y_pred_prod
df_misclassified = X_test_s[misclassified_mask].copy()
df_misclassified["True_Label"] = y_test_s[misclassified_mask]
df_misclassified["Predicted_Label"] = y_pred_prod[misclassified_mask]

print(f"Total Test Samples: {len(y_test_s)}")
print(f"Total Misclassified Samples: {len(df_misclassified)}")
print("\nSample Misclassified Instances:")
display(df_misclassified.head())

# Plot predictions confidence distribution
y_proba_prod = grid_prod.predict_proba(X_test_s)[:, 1]
axes[1].hist(
    y_proba_prod[y_test_s == 1], bins=15, alpha=0.6, label="Actual Class 1", color="teal"
)
axes[1].hist(
    y_proba_prod[y_test_s == 0], bins=15, alpha=0.6, label="Actual Class 0", color="crimson"
)
axes[1].set_xlabel("Predicted Probability (Class 1)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Predicted Class Probability Distribution")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()